# Face review by age bin
Reads `e_age.csv` (written by `pixi run -e gpu age`) and shows a mosaic for each bin, with the lowest-confidence faces first (the most likely mistakes). Just run all the cells: no widgets.

In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd
from PIL import Image
from IPython.display import display

RESULTS = Path("results")   # --out folder of the scripts (contains e_age.csv)
THUMB = 10                  # thumbnail side (px)
COLS = 10                   # thumbnails per row
MAX_PER_BIN = 10            # faces shown per bin (None = all)
ORDER_ASC = True            # True: ascending confidence (doubtful ones first)

all_rows = pd.read_csv(RESULTS / "e_age.csv")
df = all_rows[all_rows["age_bin"].notna()].copy()       # rows with a face (no-face images have an empty age_bin)
df["age_bin"] = df["age_bin"].astype(str)
bins = sorted(df["age_bin"].unique(), key=lambda b: int("".join(c for c in b.split("-")[0].split()[-1] if c.isdigit()) or 0))
display(df.groupby("age_bin").agg(n=("file", "size"), conf_mean=("age_conf", "mean"), conf_min=("age_conf", "min")).loc[bins])

In [ ]:
def thumb(r):
    img = cv2.imread(r.image_path)
    c = img[int(r.y1):int(r.y2), int(r.x1):int(r.x2)]
    c = cv2.resize(c, (THUMB, THUMB))
    cv2.rectangle(c, (0, 0), (THUMB, 16), (0, 0, 0), -1)
    cv2.putText(c, f"{r.age_conf:.0%}", (3, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    return c

def mosaic(sub):
    tiles = [thumb(r) for r in sub.itertuples()]
    while len(tiles) % COLS: tiles.append(np.zeros((THUMB, THUMB, 3), np.uint8))
    rows = [np.hstack(tiles[i:i + COLS]) for i in range(0, len(tiles), COLS)]
    return Image.fromarray(cv2.cvtColor(np.vstack(rows), cv2.COLOR_BGR2RGB))

for b in bins:
    sub = df[df.age_bin == b].sort_values("age_conf", ascending=ORDER_ASC)
    shown = sub if MAX_PER_BIN is None else sub.head(MAX_PER_BIN)
    print(f"=== {b}: {len(sub)} faces (shown {len(shown)}) ===")
    display(mosaic(shown))

## Images with no detected face (possible detector false negatives)
If they contain a face, lower `--det-score` and re-run `e_age.py` with `--overwrite`.

In [ ]:
nf = all_rows[all_rows["age_bin"].isna()].drop_duplicates("file")     # images the detector found no face in
print(f"{len(nf)} images without faces")
if len(nf):
    tiles = []
    for f in nf.image_path.head(MAX_PER_BIN or None):
        t = cv2.resize(cv2.imread(f), (THUMB, THUMB))
        tiles.append(t)
    while len(tiles) % COLS: tiles.append(np.zeros((THUMB, THUMB, 3), np.uint8))
    rows = [np.hstack(tiles[i:i + COLS]) for i in range(0, len(tiles), COLS)]
    display(Image.fromarray(cv2.cvtColor(np.vstack(rows), cv2.COLOR_BGR2RGB)))